# 🔍 RobustLens: GPU Evaluation Pipeline

This notebook runs the heavy ML evaluation for RobustLens on Kaggle GPUs.
It evaluates pretrained models on Imagenette under various distribution shifts.


In [ ]:
# 1. Setup Environment and Verify GPU
!pip install timm

import torch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! This will be very slow.")

In [ ]:
# 2. Define Dataset Mapping
import os
import urllib.request
import tarfile
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"

def download_imagenette():
    if not os.path.exists("imagenette2-320"):
        print("Downloading Imagenette...")
        urllib.request.urlretrieve(IMAGENETTE_URL, "imagenette.tgz")
        print("Extracting...")
        with tarfile.open("imagenette.tgz", "r:gz") as tar:
            tar.extractall()
        os.remove("imagenette.tgz")
    return "imagenette2-320/val"

IMAGENETTE_TO_IMAGENET = {
    "n01440764": 0,    # tench
    "n02102040": 217,  # English springer
    "n03888257": 482,  # cassette player
    "n03930630": 491,  # chain saw
    "n03028079": 405,  # church
    "n03425413": 513,  # French horn
    "n03445777": 569,  # garbage truck
    "n03445924": 571,  # gas pump
    "n03447721": 574,  # golf ball
    "n03887697": 701,  # parachute
}

class ImagenetteDataset(Dataset):
    def __init__(self, transform=None):
        val_dir = download_imagenette()
        self.dataset = ImageFolder(val_dir)
        self.transform = transform
        print("Classes found:", self.dataset.classes)
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        image, folder_idx = self.dataset[idx]
        folder_name = self.dataset.classes[folder_idx]
        imagenet_label = IMAGENETTE_TO_IMAGENET.get(folder_name, 0)
        if self.transform:
            image = self.transform(image)
        return image, imagenet_label, folder_idx

def get_imagenette_loader(batch_size=32, transform=None, subset_size=None):
    import torch
    dataset = ImagenetteDataset(transform=transform)
    if subset_size is not None and subset_size < len(dataset):
        indices = torch.randperm(len(dataset))[:subset_size]
        dataset = torch.utils.data.Subset(dataset, indices)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# 3. Run GPU Smoke Test
import time
import torchvision.transforms as T
import timm
import json
import os
import torch
import shutil
import traceback

try:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    base_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
    ])
    
    print("Loading Imagenette subset...")
    loader = get_imagenette_loader(batch_size=64, transform=base_transform, subset_size=1000)
    
    model_name = "resnet50.a1_in1k"
    print(f"\nLoading {model_name}...")
    model = timm.create_model(model_name, pretrained=True).to(device).eval()
    
    start_time = time.time()
    total_images = 0
    
    print("Running smoke test throughput benchmark...")
    with torch.no_grad():
        for images, labels, orig_labels in loader:
            images = images.to(device)
            images = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(images)
            logits = model(images)
            total_images += images.size(0)
    
    duration = time.time() - start_time
    if duration == 0: duration = 0.001
    fps = total_images / duration
    
    print(f"\n\u2705 Smoke Test Complete!")
    print(f"Device: {device}")
    print(f"Throughput: {fps:.2f} images/sec")
    print(f"Est. time for full 500k eval: {((500000 / fps) / 3600):.2f} hours")
    
    results = {
        "model": model_name,
        "throughput_fps": fps,
        "device": str(device),
        "est_hours_500k": ((500000 / fps) / 3600)
    }
    os.makedirs("/kaggle/working/results", exist_ok=True)
    with open("/kaggle/working/results/smoke_test.json", "w") as f:
        json.dump(results, f)
except Exception as e:
    with open("/kaggle/working/error.txt", "w") as f:
        f.write(traceback.format_exc())
finally:
    if os.path.exists("imagenette2-320"):
        shutil.rmtree("imagenette2-320")
